In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import (
    Input, Conv2D, MaxPooling2D, BatchNormalization,
    Activation, GlobalAveragePooling2D, GlobalMaxPooling2D,
    Concatenate, Dense, Dropout, Lambda
)
from tensorflow.keras.models import Model

# --- 1. BEMENETEK ---
input_mel    = Input(shape=(128, 1280, 1), name='mel_input')
input_chroma = Input(shape=(12,  1280, 1), name='chroma_input')
input_tempo  = Input(shape=(384, 1280, 1), name='tempo_input')

# --- 2. BRANCH ÉPÍTŐ FÜGGVÉNY ---
def create_branch(x, filters, pool_sizes, branch_name):
    for i, f in enumerate(filters):
        x = Conv2D(f, kernel_size=(3, 3), padding='same',
                   name=f'{branch_name}_conv_{i+1}')(x)
        x = BatchNormalization(name=f'{branch_name}_bn_{i+1}')(x)
        x = Activation('relu', name=f'{branch_name}_relu_{i+1}')(x)
        x = MaxPooling2D(pool_size=pool_sizes[i],
                         name=f'{branch_name}_pool_{i+1}')(x)

    avg_pool = GlobalAveragePooling2D(name=f'{branch_name}_global_avg')(x)
    max_pool = GlobalMaxPooling2D   (name=f'{branch_name}_global_max')(x)

    return Concatenate(name=f'{branch_name}_pool_concat')([avg_pool, max_pool])

# --- 3. ÁGAK ---

# MEL ÁG: (2,4) pooling az időtengely gyorsabb zsugorítása miatt
branch_mel = create_branch(
    input_mel,
    filters    = [32, 64, 128, 256],
    pool_sizes = [(2,4), (2,4), (2,2), (2,2)],   # időtengely: 1280→320→80→40→20
    branch_name = "mel"
)

# TEMPOGRAM ÁG: ugyanolyan stratégia mint a mel
branch_tempo = create_branch(
    input_tempo,
    filters    = [32, 64, 128, 256],
    pool_sizes = [(2,4), (2,4), (2,2), (2,2)],
    branch_name = "tempo"
)

# CHROMA ÁG: csak vízszintes pooling (12 hangmagasság-osztály védelme)
branch_chroma = create_branch(
    input_chroma,
    filters    = [16, 32, 64],
    pool_sizes = [(1,4), (1,4), (1,4)],           # időtengely: 1280→320→80→20
    branch_name = "chroma"
)

# --- 4. ÖSSZEFÉSÜLÉS ---
merged = Concatenate(name='concat_all_branches')(
    [branch_mel, branch_chroma, branch_tempo]
)

# --- 5. DÖNTÉSHOZÓ RÉTEGEK (BatchNorm hozzáadva) ---
z = Dense(1024, activation='relu', name='dense_1')(merged)
z = BatchNormalization(name='bn_dense_1')(z)
z = Dropout(0.4, name='dropout_1')(z)

z = Dense(512, activation='relu', name='dense_2')(z)
z = BatchNormalization(name='bn_dense_2')(z)
z = Dropout(0.4, name='dropout_2')(z)

# --- 6. KIMENET ---
# L2 normalizálás: egységnyi hosszú vektorok → cosine metrikához helyes
output_raw = Dense(400, activation='linear', name='word2vec_output')(z)
output_layer = Lambda(
    lambda x: tf.math.l2_normalize(x, axis=1),
    name='l2_norm'
)(output_raw)

# --- 7. LOSS: 1 - cosine_similarity (Keras minimalizál, ezért kell a negálás) ---
def cosine_loss(y_true, y_pred):
    # cosine_similarity() értéke [-1, 1], negálva minimalizálható
    return 1.0 - tf.reduce_mean(
        tf.reduce_sum(
            tf.math.l2_normalize(y_true, axis=1) * y_pred,
            axis=1
        )
    )

# --- 8. ÖSSZESZEREÉS ÉS FORDÍTÁS ---
model = Model(
    inputs  = [input_mel, input_chroma, input_tempo],
    outputs = output_layer
)

model.compile(
    optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss      = cosine_loss,
    metrics   = ['mae']
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ mel_input           │ (None, 128, 1280, │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tempo_input         │ (None, 384, 1280, │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mel_conv_1 (Conv2D) │ (None, 128, 1280, │        320 │ mel_input[0][0]   │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tempo_conv_1        │ (None, 384, 1280, │        320 │ tempo_input[0][0] │
│ (Conv2D)            │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mel_bn_1            │ (None, 128, 1280, │        128 │ mel_conv_1[0][0]  │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tempo_bn_1          │ (None, 384, 1280, │        128 │ tempo_conv_1[0][… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mel_relu_1          │ (None, 128, 1280, │          0 │ mel_bn_1[0][0]    │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tempo_relu_1        │ (None, 384, 1280, │          0 │ tempo_bn_1[0][0]  │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mel_pool_1          │ (None, 64, 320,   │          0 │ mel_relu_1[0][0]  │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ chroma_input        │ (None, 12, 1280,  │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tempo_pool_1        │ (None, 192, 320,  │          0 │ tempo_relu_1[0][… │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mel_conv_2 (Conv2D) │ (None, 64, 320,   │     18,496 │ mel_pool_1[0][0]  │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ chroma_conv_1       │ (None, 12, 1280,  │        160 │ chroma_input[0][… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tempo_conv_2        │ (None, 192, 320,  │     18,496 │ tempo_pool_1[0][… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mel_bn_2            │ (None, 64, 320,   │        256 │ mel_conv_2[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ chroma_bn_1         │ (None, 12, 1280,  │         64 │ chroma_conv_1[0]… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tempo_bn_2          │ (None, 192, 320,  │        256 │ tempo_conv_2[0][

 Total params: 2,720,080 (10.38 MB)

 Trainable params: 2,714,864 (10.36 MB)

 Non-trainable params: 5,216 (20.38 KB)

In [ ]:
import os

import h5py
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import (
    Input, Conv2D, MaxPooling2D, BatchNormalization,
    Activation, GlobalAveragePooling2D, GlobalMaxPooling2D,
    Concatenate, Dense, Dropout
)
from tensorflow.keras.models import Model
from tensorflow.keras.utils import Sequence
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from gensim.models import Word2Vec

# --- ÚTVONALAK A COLABON ---
H5_PATH = "./Dataset/spotify_dataset_compressed.h5"
W2V_PATH = "./song2vec.model" # Frissítve a te fájlnevedre!
SAVE_PATH = "./spotify_cnn_model.keras"

# ==========================================
# 1. A PINCÉR: DATA GENERATOR
# ==========================================
class SpotifyDataGenerator(Sequence):
    def __init__(self, h5_path, w2v_model, split_name='train', batch_size=64, shuffle=True):
        self.h5_path = h5_path
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.w2v_model = w2v_model

        with h5py.File(self.h5_path, 'r') as hf:
            splits = hf['ml/split'][:]
            splits_str = np.array([s.decode('utf-8') if isinstance(s, bytes) else s for s in splits])
            self.indices = np.where(splits_str == split_name)[0]

        print(f"✅ Generátor '{split_name}': {len(self.indices)} dal (Batch size: {batch_size}).")
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.indices) / self.batch_size))

    def __getitem__(self, index):
        batch_indices = self.indices[index * self.batch_size : (index + 1) * self.batch_size]
        batch_indices = np.sort(batch_indices)

        with h5py.File(self.h5_path, 'r') as hf:
            X_mel = hf['spectrograms/mel'][batch_indices]
            X_chroma = hf['spectrograms/chroma'][batch_indices]
            X_tempo = hf['spectrograms/tempogram'][batch_indices]
            uris = hf['ml/track_uri'][batch_indices]

        # Csatorna dimenzió hozzáadása
        X_mel = np.expand_dims(X_mel, axis=-1)
        X_chroma = np.expand_dims(X_chroma, axis=-1)
        X_tempo = np.expand_dims(X_tempo, axis=-1)

        y_batch = np.zeros((len(batch_indices), 400), dtype=np.float32)
        for i, uri_bytes in enumerate(uris):
            uri = uri_bytes.decode('utf-8') if isinstance(uri_bytes, bytes) else uri_bytes
            if uri in self.w2v_model.wv:
                y_batch[i] = self.w2v_model.wv[uri]

        # A JAVÍTÁS: List helyett Tuple-t adunk vissza a bemeneteknél
        # Eredeti: return [X_mel, X_chroma, X_tempo], y_batch
        return (X_mel, X_chroma, X_tempo), y_batch

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

class L2NormLayer(tf.keras.layers.Layer):
    def call(self, x):
        return tf.math.l2_normalize(x, axis=1)

# ==========================================
# 2. AZ AGY: MODELL ARCHITEKTÚRA
# ==========================================
def create_branch(x, filters, pool_sizes, branch_name):
    for i, f in enumerate(filters):
        x = Conv2D(f, kernel_size=(3, 3), padding='same', name=f'{branch_name}_conv_{i+1}')(x)
        x = BatchNormalization(name=f'{branch_name}_bn_{i+1}')(x)
        x = Activation('relu', name=f'{branch_name}_relu_{i+1}')(x)
        x = MaxPooling2D(pool_size=pool_sizes[i], name=f'{branch_name}_pool_{i+1}')(x)
    avg_pool = GlobalAveragePooling2D(name=f'{branch_name}_global_avg')(x)
    max_pool = GlobalMaxPooling2D(name=f'{branch_name}_global_max')(x)
    return Concatenate(name=f'{branch_name}_pool_concat')([avg_pool, max_pool])

def cosine_loss(y_true, y_pred):
    return 1.0 - tf.reduce_mean(tf.reduce_sum(tf.math.l2_normalize(y_true, axis=1) * y_pred, axis=1))

print("\n🚀 Modell építése...")
input_mel    = Input(shape=(128, 1280, 1), name='mel_input')
input_chroma = Input(shape=(12,  1280, 1), name='chroma_input')
input_tempo  = Input(shape=(384, 1280, 1), name='tempo_input')

branch_mel = create_branch(input_mel, [32, 64, 128, 256], [(2,4), (2,4), (2,2), (2,2)], "mel")
branch_tempo = create_branch(input_tempo, [32, 64, 128, 256], [(2,4), (2,4), (2,2), (2,2)], "tempo")
branch_chroma = create_branch(input_chroma, [16, 32, 64], [(1,4), (1,4), (1,4)], "chroma")

merged = Concatenate(name='concat_all_branches')([branch_mel, branch_chroma, branch_tempo])
z = Dense(1024, activation='relu', name='dense_1')(merged)
z = BatchNormalization(name='bn_dense_1')(z)
z = Dropout(0.4, name='dropout_1')(z)
z = Dense(512, activation='relu', name='dense_2')(z)
z = BatchNormalization(name='bn_dense_2')(z)
z = Dropout(0.4, name='dropout_2')(z)

output_raw = Dense(400, activation='linear', name='word2vec_output')(z)
output_layer = L2NormLayer(name='l2_norm')(output_raw)

model = Model(inputs=[input_mel, input_chroma, input_tempo], outputs=output_layer)
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3), loss=cosine_loss, metrics=['mae'])

# ==========================================
# 3. A STARTPISZTOLY: TANÍTÁS
# ==========================================
print("\n📚 Word2Vec betöltése...")
w2v_model = Word2Vec.load(W2V_PATH)

print("\n⚙️ Generátorok inicializálása...")
train_gen = SpotifyDataGenerator(H5_PATH, w2v_model, split_name='train', batch_size=64)
val_gen = SpotifyDataGenerator(H5_PATH, w2v_model, split_name='val', batch_size=64, shuffle=False)

# --- SÚLYOK BETÖLTÉSE (BIZTONSÁGI MÓDSZER) ---
if os.path.exists(SAVE_PATH):
    print(f"\n💾 Korábbi állapot megtalálva!")
    model = tf.keras.models.load_model(
        SAVE_PATH,
        custom_objects={
            'cosine_loss': cosine_loss,
            'L2NormLayer': L2NormLayer
        }
    )
    print("✅ Modell sikeresen betöltve!")
else:
    print(f"\n⚠️ FIGYELEM: Nem található fájl a {SAVE_PATH} útvonalon!")
    print("Indulás a nulláról.")

callbacks = [
    # A legjobb modellt mindig egyből kimenti a Google Drive-odra!
    ModelCheckpoint(SAVE_PATH, save_best_only=True, monitor='val_loss', mode='min', verbose=1),
    # Ha 5 körön át nem javul, leállítja magát
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1)
]

print("\n🔥 GPU TANÍTÁS FOLYTATÁSA...")
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=20,          
    initial_epoch=8,    # Megmondjuk neki, hogy a 7. már kész van!
    callbacks=callbacks
)


🚀 Modell építése...

📚 Word2Vec betöltése...

⚙️ Generátorok inicializálása...
✅ Generátor 'train': 21642 dal (Batch size: 64).
✅ Generátor 'val': 2705 dal (Batch size: 64).

💾 Korábbi állapot megtalálva!
✅ Modell sikeresen betöltve!

🔥 GPU TANÍTÁS FOLYTATÁSA...


c:\Users\Béres Gábor\music_recommender\.venv\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 9/20


In [ ]:
pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 66.8 MB/s eta 0:00:00:00:0100:01


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
pip install tensorflow

  Using cached tensorflow-2.21.0-cp312-cp312-win_amd64.whl.metadata (4.5 kB)
Using cached tensorflow-2.21.0-cp312-cp312-win_amd64.whl (350.9 MB)
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Mennyire jók a Word2Vec vektorok önmagukban?
def w2v_only_auc(y_true, n=1000):
    scores, labels = [], []
    idx = np.random.choice(len(y_true), min(n, len(y_true)), replace=False)
    for i in idx:
        p_sim = np.dot(y_true[i], y_true[i])  # tökéletes egyezés
        scores.append(p_sim); labels.append(1)
        ni = np.random.randint(0, len(y_true))
        while ni == i: ni = np.random.randint(0, len(y_true))
        n_sim = np.dot(y_true[i], y_true[ni])
        scores.append(n_sim); labels.append(0)
    return roc_auc_score(labels, scores)

print(f"Word2Vec önmagában: {w2v_only_auc(y_true):.4f}")

Word2Vec önmagában: 0.9980


# Összehasonlítható modell

### Releváns uri változás

In [5]:
import h5py
import numpy as np
import tensorflow as tf
from gensim.models import Word2Vec
from collections import defaultdict
from tqdm import tqdm 

# --- ÚTVONALAK ÉS BEÁLLÍTÁSOK ---
H5_PATH = "../Dataset/spotify_dataset.h5"
W2V_PATH = "../Models/song2vec.model"
CNN_PATH = "../Models/spotify_cnn_model.keras" 
K_VALUES = [10, 20, 50, 100, 500] 
NUM_TEST_SAMPLES = 500 # Állítsd None-ra, ha az összes "minden 10." dalt tesztelni akarod!
STEP_SIZE = 5 # <--- ÚJ BEÁLLÍTÁS: Minden hanyadik elemet vegyük

# Custom rétegek és loss a CNN betöltéséhez
class L2NormLayer(tf.keras.layers.Layer):
    def call(self, x):
        return tf.math.l2_normalize(x, axis=1)

def cosine_loss(y_true, y_pred):
    return 1.0 - tf.reduce_mean(tf.reduce_sum(tf.math.l2_normalize(y_true, axis=1) * y_pred, axis=1))

def calculate_metrics(recommended_uris, relevant_uris, K):
    top_k = recommended_uris[:K]
    hits = len(set(top_k).intersection(relevant_uris))
    
    hit_rate = 1 if hits > 0 else 0
    precision = hits / K
    recall = hits / len(relevant_uris) if len(relevant_uris) > 0 else 0
    
    return hit_rate, precision, recall

def main():
    print("1. Modellek betöltése...")
    w2v_model = Word2Vec.load(W2V_PATH)
    cnn_model = tf.keras.models.load_model(
        CNN_PATH, custom_objects={'cosine_loss': cosine_loss, 'L2NormLayer': L2NormLayer}
    )

    print("\n2. Ground Truth szótárak építése a RAM-ban (ez eltarthat 1-2 percig)...")
    track_to_playlists = defaultdict(set)
    playlist_to_tracks = defaultdict(set)
    
    with h5py.File(H5_PATH, "r") as hf:
        pids = hf["playlist_tracks/pid"][:]
        uris = hf["playlist_tracks/track_uri"][:]
        
        for i in tqdm(range(len(pids)), desc="Kapcsolatok feldolgozása"):
            pid = pids[i]
            uri = uris[i].decode('utf-8') if isinstance(uris[i], bytes) else uris[i]
            
            track_to_playlists[uri].add(pid)
            playlist_to_tracks[pid].add(uri)

    print("\n3. Teszt halmaz kinyerése...")
    with h5py.File(H5_PATH, "r") as hf:
        splits = hf["ml/split"][:]
        # Először kigyűjtjük az ÖSSZES "test" indexet
        all_test_indices = [i for i, s in enumerate(splits) if (s.decode('utf-8') if isinstance(s, bytes) else s) == "test"]
        
        # ---> ÚJ LOGIKA: Minden 10. elem kiválasztása determinisztikusan <---
        test_indices = all_test_indices[::STEP_SIZE]
        
        # Ha van limit (pl. csak 500-at akarunk futtatni a gyorsaság miatt), akkor levágjuk az elejét
        if NUM_TEST_SAMPLES and len(test_indices) > NUM_TEST_SAMPLES:
            test_indices = test_indices[:NUM_TEST_SAMPLES]
            
        print(f"Tesztelésre kiválasztva: {len(test_indices)} dal (minden {STEP_SIZE}. teszt elemből).")

        results = {"baseline": {k: {"hr": [], "prec": [], "rec": []} for k in K_VALUES},
                   "cnn":      {k: {"hr": [], "prec": [], "rec": []} for k in K_VALUES}}

        print("\n4. Kiértékelés futtatása...")
        for idx in tqdm(test_indices, desc="Dalok tesztelése"):
            uri = hf["ml/track_uri"][idx]
            uri = uri.decode('utf-8') if isinstance(uri, bytes) else uri
            
            if uri not in w2v_model.wv:
                continue
                
            # --- GROUND TRUTH KINYERÉSE ---
            pids = track_to_playlists.get(uri, set())
            if not pids: 
                continue 
                
            relevant_uris = set()
            for pid in pids:
                relevant_uris.update(playlist_to_tracks[pid])
            relevant_uris.discard(uri) 
            
            # Csak az marad a relevánsak között, amihez van w2v vektor
            relevant_uris = {u for u in relevant_uris if u in w2v_model.wv}
            
            if not relevant_uris:
                continue

            # --- A) BASELINE: Sima Word2Vec predikció ---
            w2v_similars = w2v_model.wv.most_similar(uri, topn=max(K_VALUES))
            w2v_recs = [u for u, score in w2v_similars]

            # --- B) CNN MODELL PREDIKCIÓ ---
            mel = np.expand_dims(np.expand_dims(hf["spectrograms/mel"][idx], axis=0), axis=-1)
            chroma = np.expand_dims(np.expand_dims(hf["spectrograms/chroma"][idx], axis=0), axis=-1)
            tempo = np.expand_dims(np.expand_dims(hf["spectrograms/tempogram"][idx], axis=0), axis=-1)
            
            pred_vector = cnn_model.predict([mel, chroma, tempo], verbose=0)[0]
            
            cnn_similars = w2v_model.wv.similar_by_vector(pred_vector, topn=max(K_VALUES))
            cnn_recs = [u for u, score in cnn_similars if u != uri][:max(K_VALUES)]
            
            if len(cnn_recs) < max(K_VALUES):
                extra = w2v_model.wv.similar_by_vector(pred_vector, topn=max(K_VALUES)+1)
                cnn_recs = [u for u, score in extra if u != uri][:max(K_VALUES)]

            # --- METRIKÁK KISZÁMÍTÁSA ---
            for k in K_VALUES:
                hr_b, prec_b, rec_b = calculate_metrics(w2v_recs, relevant_uris, k)
                results["baseline"][k]["hr"].append(hr_b)
                results["baseline"][k]["prec"].append(prec_b)
                results["baseline"][k]["rec"].append(rec_b)
                
                hr_c, prec_c, rec_c = calculate_metrics(cnn_recs, relevant_uris, k)
                results["cnn"][k]["hr"].append(hr_c)
                results["cnn"][k]["prec"].append(prec_c)
                results["cnn"][k]["rec"].append(rec_c)

    # --- 5. EREDMÉNYEK KIÍRÁSA ---
    print("\n" + "="*50)
    print("🏆 KIÉRTÉKELÉS EREDMÉNYE 🏆")
    print("="*50)
    for k in K_VALUES:
        print(f"\n--- TOP {k} AJÁNLÁS ---")
        print("BASELINE (Tiszta Word2Vec - Felső korlát):")
        print(f"  Hit Rate@{k}:  {np.mean(results['baseline'][k]['hr'])*100:.2f}%")
        print(f"  Precision@{k}: {np.mean(results['baseline'][k]['prec'])*100:.2f}%")
        print(f"  Recall@{k}:    {np.mean(results['baseline'][k]['rec'])*100:.2f}%")
        
        print("\nCNN MODELL (Audio alapján generált vektor):")
        print(f"  Hit Rate@{k}:  {np.mean(results['cnn'][k]['hr'])*100:.2f}%")
        print(f"  Precision@{k}: {np.mean(results['cnn'][k]['prec'])*100:.2f}%")
        print(f"  Recall@{k}:    {np.mean(results['cnn'][k]['rec'])*100:.2f}%")

if __name__ == "__main__":
    main()

1. Modellek betöltése...

2. Ground Truth szótárak építése a RAM-ban (ez eltarthat 1-2 percig)...


Kapcsolatok feldolgozása: 100%|██████████| 66346428/66346428 [02:50<00:00, 389552.92it/s]



3. Teszt halmaz kinyerése...
Tesztelésre kiválasztva: 500 dal (minden 5. teszt elemből).

4. Kiértékelés futtatása...


Dalok tesztelése: 100%|██████████| 500/500 [03:32<00:00,  2.35it/s]



🏆 KIÉRTÉKELÉS EREDMÉNYE 🏆

--- TOP 10 AJÁNLÁS ---
BASELINE (Tiszta Word2Vec - Felső korlát):
  Hit Rate@10:  98.40%
  Precision@10: 84.20%
  Recall@10:    0.15%

CNN MODELL (Audio alapján generált vektor):
  Hit Rate@10:  55.20%
  Precision@10: 19.48%
  Recall@10:    0.01%

--- TOP 20 AJÁNLÁS ---
BASELINE (Tiszta Word2Vec - Felső korlát):
  Hit Rate@20:  98.80%
  Precision@20: 80.85%
  Recall@20:    0.27%

CNN MODELL (Audio alapján generált vektor):
  Hit Rate@20:  65.40%
  Precision@20: 19.40%
  Recall@20:    0.03%

--- TOP 50 AJÁNLÁS ---
BASELINE (Tiszta Word2Vec - Felső korlát):
  Hit Rate@50:  99.80%
  Precision@50: 75.26%
  Recall@50:    0.61%

CNN MODELL (Audio alapján generált vektor):
  Hit Rate@50:  75.20%
  Precision@50: 19.48%
  Recall@50:    0.08%

--- TOP 100 AJÁNLÁS ---
BASELINE (Tiszta Word2Vec - Felső korlát):
  Hit Rate@100:  100.00%
  Precision@100: 70.36%
  Recall@100:    1.08%

CNN MODELL (Audio alapján generált vektor):
  Hit Rate@100:  81.80%
  Precision@100: 19.

# AUC-t is számolok

In [1]:
import h5py
import numpy as np
import tensorflow as tf
from gensim.models import Word2Vec
from collections import defaultdict
from tqdm import tqdm 
import random
from sklearn.metrics import roc_auc_score

# ==========================================
# 1. ÚTVONALAK ÉS BEÁLLÍTÁSOK
# ==========================================
H5_PATH = "../Dataset/spotify_dataset.h5"
W2V_PATH = "../Models/song2vec.model"
CNN_PATH = "../Models/spotify_cnn_model.keras" 
K_VALUES = [10, 20, 50, 100, 500] 

# Tesztelési korlátok a gyorsasághoz
NUM_TEST_SAMPLES = 500  # None, ha az összeset akarod
STEP_SIZE = 5           # Minden hanyadik teszt elemet vegyük

# ==========================================
# 2. SEGÉDFÜGGVÉNYEK ÉS EGYEDI RÉTEGEK
# ==========================================
class L2NormLayer(tf.keras.layers.Layer):
    def call(self, x):
        return tf.math.l2_normalize(x, axis=1)

def cosine_loss(y_true, y_pred):
    return 1.0 - tf.reduce_mean(tf.reduce_sum(tf.math.l2_normalize(y_true, axis=1) * y_pred, axis=1))

def calculate_metrics(recommended_uris, relevant_uris, K):
    top_k = recommended_uris[:K]
    hits = len(set(top_k).intersection(relevant_uris))
    
    hit_rate = 1 if hits > 0 else 0
    precision = hits / K
    recall = hits / len(relevant_uris) if len(relevant_uris) > 0 else 0
    
    return hit_rate, precision, recall

# --- AUC SZÁMÍTÁSA (Sampled ROC-AUC) ---
def calculate_auc(query_vector, relevant_uris, w2v_model, num_negatives=500):
    """
    Megméri, hogy a modell mekkora eséllyel rangsorolja előrébb a 
    valóban releváns dalokat a véletlen negatívokhoz képest.
    """
    # 1. Pozitív minták pontszámai (amikkel már volt közös listán)
    pos_scores = []
    for uri in relevant_uris:
        if uri in w2v_model.wv:
            # Koszinusz hasonlóság (vektorok normáltak, így a dot product elég)
            score = np.dot(query_vector, w2v_model.wv[uri])
            pos_scores.append(score)
    
    if not pos_scores: 
        return 0.5 # Ha nincs pozitív adat, a véletlen szintet adjuk vissza
    
    # 2. Negatív minták (véletlen dalok, amikkel SOHA nem volt közös listán)
    neg_scores = []
    all_uris = list(w2v_model.wv.key_to_index.keys())
    
    # Biztonsági korlát a mintavételhez
    n_neg = min(num_negatives, len(all_uris) - len(relevant_uris))
    
    while len(neg_scores) < n_neg:
        random_uri = random.choice(all_uris)
        if random_uri not in relevant_uris:
            score = np.dot(query_vector, w2v_model.wv[random_uri])
            neg_scores.append(score)
            
    # 3. ROC-AUC kiszámítása a scikit-learn segítségével
    y_true = [1] * len(pos_scores) + [0] * len(neg_scores)
    y_scores = pos_scores + neg_scores
    
    try:
        return roc_auc_score(y_true, y_scores)
    except:
        return 0.5

# ==========================================
# 3. FŐPROGRAM
# ==========================================
def main():
    print("1. Modellek betöltése...")
    w2v_model = Word2Vec.load(W2V_PATH)
    cnn_model = tf.keras.models.load_model(
        CNN_PATH, custom_objects={'cosine_loss': cosine_loss, 'L2NormLayer': L2NormLayer}
    )

    print("\n2. Kapcsolati háló (Ground Truth) építése a memóriában...")
    track_to_playlists = defaultdict(set)
    playlist_to_tracks = defaultdict(set)
    
    with h5py.File(H5_PATH, "r") as hf:
        pids = hf["playlist_tracks/pid"][:]
        uris = hf["playlist_tracks/track_uri"][:]
        
        for i in tqdm(range(len(pids)), desc="Kapcsolatok feldolgozása"):
            pid = pids[i]
            uri = uris[i].decode('utf-8') if isinstance(uris[i], bytes) else uris[i]
            
            track_to_playlists[uri].add(pid)
            playlist_to_tracks[pid].add(uri)

    print("\n3. Teszt halmaz előkészítése...")
    with h5py.File(H5_PATH, "r") as hf:
        splits = hf["ml/split"][:]
        all_test_indices = [i for i, s in enumerate(splits) if (s.decode('utf-8') if isinstance(s, bytes) else s) == "test"]
        
        # Determinisztikus mintavétel
        test_indices = all_test_indices[::STEP_SIZE]
        
        if NUM_TEST_SAMPLES and len(test_indices) > NUM_TEST_SAMPLES:
            test_indices = test_indices[:NUM_TEST_SAMPLES]
            
        print(f"Tesztelésre kijelölve: {len(test_indices)} dal.")

        # Eredménytároló szótár
        results = {
            "baseline": {"auc": [], **{k: {"hr": [], "prec": [], "rec": []} for k in K_VALUES}},
            "cnn":      {"auc": [], **{k: {"hr": [], "prec": [], "rec": []} for k in K_VALUES}}
        }

        print("\n4. Kiértékelés indítása (AUC + Top-K)...")
        for idx in tqdm(test_indices, desc="Dalok tesztelése"):
            uri = hf["ml/track_uri"][idx]
            uri = uri.decode('utf-8') if isinstance(uri, bytes) else uri
            
            if uri not in w2v_model.wv:
                continue
                
            # --- RELEVÁNS DALOK (Akikkel már volt egy listán) ---
            pids_of_track = track_to_playlists.get(uri, set())
            if not pids_of_track: continue 
                
            relevant_uris = set()
            for pid in pids_of_track:
                relevant_uris.update(playlist_to_tracks[pid])
            
            relevant_uris.discard(uri) # Önmagát ne ajánlja
            relevant_uris = {u for u in relevant_uris if u in w2v_model.wv} # Csak ami benne van a szótárban
            
            if not relevant_uris: continue

            # --- A) BASELINE (Word2Vec) ---
            baseline_vector = w2v_model.wv[uri]
            w2v_similars = w2v_model.wv.most_similar(uri, topn=max(K_VALUES))
            w2v_recs = [u for u, score in w2v_similars]

            # --- B) CNN MODELL (Audio) ---
            mel = np.expand_dims(np.expand_dims(hf["spectrograms/mel"][idx], axis=0), axis=-1)
            chroma = np.expand_dims(np.expand_dims(hf["spectrograms/chroma"][idx], axis=0), axis=-1)
            tempo = np.expand_dims(np.expand_dims(hf["spectrograms/tempogram"][idx], axis=0), axis=-1)
            
            pred_vector = cnn_model.predict([mel, chroma, tempo], verbose=0)[0]
            
            cnn_similars = w2v_model.wv.similar_by_vector(pred_vector, topn=max(K_VALUES)+1)
            cnn_recs = [u for u, score in cnn_similars if u != uri][:max(K_VALUES)]

            # --- AUC SZÁMÍTÁS ---
            results["baseline"]["auc"].append(calculate_auc(baseline_vector, relevant_uris, w2v_model))
            results["cnn"]["auc"].append(calculate_auc(pred_vector, relevant_uris, w2v_model))

            # --- TOP-K METRIKÁK SZÁMÍTÁSA ---
            for k in K_VALUES:
                # Baseline
                hr_b, prec_b, rec_b = calculate_metrics(w2v_recs, relevant_uris, k)
                results["baseline"][k]["hr"].append(hr_b)
                results["baseline"][k]["prec"].append(prec_b)
                results["baseline"][k]["rec"].append(rec_b)
                
                # CNN
                hr_c, prec_c, rec_c = calculate_metrics(cnn_recs, relevant_uris, k)
                results["cnn"][k]["hr"].append(hr_c)
                results["cnn"][k]["prec"].append(prec_c)
                results["cnn"][k]["rec"].append(rec_c)

    # ==========================================
    # 4. EREDMÉNYEK ÖSSZESÍTÉSE
    # ==========================================
    print("\n" + "="*60)
    print("🏆 VÉGLEGES KIÉRTÉKELÉS (Item-to-Item Similarity) 🏆")
    print("="*60)
    
    print(f"\n✨ GLOBÁLIS RANGSOROLÁSI MINŐSÉG:")
    print(f"  BASELINE (Word2Vec) AUC: {np.mean(results['baseline']['auc'])*100:.2f}%")
    print(f"  CNN MODEL (Audio)    AUC: {np.mean(results['cnn']['auc'])*100:.2f}%")

    for k in K_VALUES:
        print(f"\n--- TOP {k} AJÁNLÁS ---")
        print(f"  BASELINE -> HR: {np.mean(results['baseline'][k]['hr'])*100:.2f}% | Prec: {np.mean(results['baseline'][k]['prec'])*100:.2f}% | Rec: {np.mean(results['baseline'][k]['rec'])*100:.2f}%")
        print(f"  CNN MODELL -> HR: {np.mean(results['cnn'][k]['hr'])*100:.2f}% | Prec: {np.mean(results['cnn'][k]['prec'])*100:.2f}% | Rec: {np.mean(results['cnn'][k]['rec'])*100:.2f}%")
    print("\n" + "="*60)

if __name__ == "__main__":
    main()

1. Modellek betöltése...


2. Kapcsolati háló (Ground Truth) építése a memóriában...


Kapcsolatok feldolgozása: 100%|██████████| 66346428/66346428 [02:23<00:00, 463205.93it/s]



3. Teszt halmaz előkészítése...
Tesztelésre kijelölve: 500 dal.

4. Kiértékelés indítása (AUC + Top-K)...


Dalok tesztelése: 100%|██████████| 500/500 [03:24<00:00,  2.45it/s]



🏆 VÉGLEGES KIÉRTÉKELÉS (Item-to-Item Similarity) 🏆

✨ GLOBÁLIS RANGSOROLÁSI MINŐSÉG:
  BASELINE (Word2Vec) AUC: 82.42%
  CNN MODEL (Audio)    AUC: 73.77%

--- TOP 10 AJÁNLÁS ---
  BASELINE -> HR: 98.40% | Prec: 84.20% | Rec: 0.15%
  CNN MODELL -> HR: 55.20% | Prec: 19.48% | Rec: 0.01%

--- TOP 20 AJÁNLÁS ---
  BASELINE -> HR: 98.80% | Prec: 80.85% | Rec: 0.27%
  CNN MODELL -> HR: 65.40% | Prec: 19.40% | Rec: 0.03%

--- TOP 50 AJÁNLÁS ---
  BASELINE -> HR: 99.80% | Prec: 75.26% | Rec: 0.61%
  CNN MODELL -> HR: 75.20% | Prec: 19.48% | Rec: 0.08%

--- TOP 100 AJÁNLÁS ---
  BASELINE -> HR: 100.00% | Prec: 70.36% | Rec: 1.08%
  CNN MODELL -> HR: 81.80% | Prec: 19.37% | Rec: 0.16%

--- TOP 500 AJÁNLÁS ---
  BASELINE -> HR: 100.00% | Prec: 56.66% | Rec: 3.63%
  CNN MODELL -> HR: 91.80% | Prec: 18.96% | Rec: 0.80%



In [3]:
import h5py
import numpy as np
import tensorflow as tf
from gensim.models import Word2Vec
from collections import defaultdict
from tqdm import tqdm

# --- ÚTVONALAK ÉS BEÁLLÍTÁSOK ---
H5_PATH = "../Dataset/spotify_dataset.h5"
W2V_PATH = "../Models/song2vec.model"
CNN_PATH = "../Models/spotify_cnn_model.keras" 
K_VALUES = [10, 20] 
NUM_TEST_SAMPLES = 500 

# Custom rétegek és loss a CNN betöltéséhez
class L2NormLayer(tf.keras.layers.Layer):
    def call(self, x):
        return tf.math.l2_normalize(x, axis=1)

def cosine_loss(y_true, y_pred):
    return 1.0 - tf.reduce_mean(tf.reduce_sum(tf.math.l2_normalize(y_true, axis=1) * y_pred, axis=1))

def calculate_metrics(recommended_uris, relevant_uris, K):
    top_k = recommended_uris[:K]
    hits = len(set(top_k).intersection(relevant_uris))
    
    hit_rate = 1 if hits > 0 else 0
    precision = hits / K
    recall = hits / len(relevant_uris) if len(relevant_uris) > 0 else 0
    
    return hit_rate, precision, recall

def main():
    print("1. Modellek betöltése...")
    w2v_model = Word2Vec.load(W2V_PATH)
    cnn_model = tf.keras.models.load_model(
        CNN_PATH, custom_objects={'cosine_loss': cosine_loss, 'L2NormLayer': L2NormLayer}
    )

    print("\n2. Ground Truth szótárak építése a RAM-ban (ez eltarthat 1-2 percig)...")
    track_to_playlists = defaultdict(set)
    playlist_to_tracks = defaultdict(set)
    
    with h5py.File(H5_PATH, "r") as hf:
        pids = hf["playlist_tracks/pid"][:]
        uris = hf["playlist_tracks/track_uri"][:]
        
        for i in tqdm(range(len(pids)), desc="Kapcsolatok feldolgozása"):
            pid = pids[i]
            uri = uris[i].decode('utf-8') if isinstance(uris[i], bytes) else uris[i]
            
            track_to_playlists[uri].add(pid)
            playlist_to_tracks[pid].add(uri)

    print("\n3. Teszt halmaz kinyerése...")
    with h5py.File(H5_PATH, "r") as hf:
        splits = hf["ml/split"][:]
        test_indices = [i for i, s in enumerate(splits) if (s.decode('utf-8') if isinstance(s, bytes) else s) == "test"]
        
        if NUM_TEST_SAMPLES and len(test_indices) > NUM_TEST_SAMPLES:
            np.random.seed(42)
            test_indices = np.random.choice(test_indices, NUM_TEST_SAMPLES, replace=False)
            
        print(f"Tesztelésre kiválasztva: {len(test_indices)} dal.")

        results = {"baseline": {k: {"hr": [], "prec": [], "rec": []} for k in K_VALUES},
                   "cnn":      {k: {"hr": [], "prec": [], "rec": []} for k in K_VALUES}}

        print("\n4. Kiértékelés futtatása...")
        for idx in tqdm(test_indices, desc="Dalok tesztelése"):
            uri = hf["ml/track_uri"][idx]
            uri = uri.decode('utf-8') if isinstance(uri, bytes) else uri
            
            if uri not in w2v_model.wv:
                continue
                
            # --- GROUND TRUTH KINYERÉSE ---
            pids = track_to_playlists.get(uri, set())
            if not pids: 
                continue 
                
            relevant_uris = set()
            for pid in pids:
                relevant_uris.update(playlist_to_tracks[pid])
            relevant_uris.discard(uri) 
            
            # ---> ÚJ SZŰRÉS: Csak az marad a relevánsak között, amihez van w2v vektor <---
            relevant_uris = {u for u in relevant_uris if u in w2v_model.wv}
            
            if not relevant_uris:
                continue

            # --- A) BASELINE: Sima Word2Vec predikció ---
            w2v_similars = w2v_model.wv.most_similar(uri, topn=max(K_VALUES))
            w2v_recs = [u for u, score in w2v_similars]

            # --- B) CNN MODELL PREDIKCIÓ ---
            mel = np.expand_dims(np.expand_dims(hf["spectrograms/mel"][idx], axis=0), axis=-1)
            chroma = np.expand_dims(np.expand_dims(hf["spectrograms/chroma"][idx], axis=0), axis=-1)
            tempo = np.expand_dims(np.expand_dims(hf["spectrograms/tempogram"][idx], axis=0), axis=-1)
            
            pred_vector = cnn_model.predict([mel, chroma, tempo], verbose=0)[0]
            
            cnn_similars = w2v_model.wv.similar_by_vector(pred_vector, topn=max(K_VALUES))
            cnn_recs = [u for u, score in cnn_similars if u != uri][:max(K_VALUES)]
            
            if len(cnn_recs) < max(K_VALUES):
                extra = w2v_model.wv.similar_by_vector(pred_vector, topn=max(K_VALUES)+1)
                cnn_recs = [u for u, score in extra if u != uri][:max(K_VALUES)]

            # --- METRIKÁK KISZÁMÍTÁSA ---
            for k in K_VALUES:
                hr_b, prec_b, rec_b = calculate_metrics(w2v_recs, relevant_uris, k)
                results["baseline"][k]["hr"].append(hr_b)
                results["baseline"][k]["prec"].append(prec_b)
                results["baseline"][k]["rec"].append(rec_b)
                
                hr_c, prec_c, rec_c = calculate_metrics(cnn_recs, relevant_uris, k)
                results["cnn"][k]["hr"].append(hr_c)
                results["cnn"][k]["prec"].append(prec_c)
                results["cnn"][k]["rec"].append(rec_c)

    # --- 5. EREDMÉNYEK KIÍRÁSA ---
    print("\n" + "="*50)
    print("🏆 KIÉRTÉKELÉS EREDMÉNYE 🏆")
    print("="*50)
    for k in K_VALUES:
        print(f"\n--- TOP {k} AJÁNLÁS ---")
        print("BASELINE (Tiszta Word2Vec - Felső korlát):")
        print(f"  Hit Rate@{k}:  {np.mean(results['baseline'][k]['hr'])*100:.2f}%")
        print(f"  Precision@{k}: {np.mean(results['baseline'][k]['prec'])*100:.2f}%")
        print(f"  Recall@{k}:    {np.mean(results['baseline'][k]['rec'])*100:.2f}%")
        
        print("\nCNN MODELL (Audio alapján generált vektor):")
        print(f"  Hit Rate@{k}:  {np.mean(results['cnn'][k]['hr'])*100:.2f}%")
        print(f"  Precision@{k}: {np.mean(results['cnn'][k]['prec'])*100:.2f}%")
        print(f"  Recall@{k}:    {np.mean(results['cnn'][k]['rec'])*100:.2f}%")

if __name__ == "__main__":
    main()

1. Modellek betöltése...

2. Ground Truth szótárak építése a RAM-ban (ez eltarthat 1-2 percig)...


Kapcsolatok feldolgozása: 100%|██████████| 66346428/66346428 [02:11<00:00, 505050.37it/s]



3. Teszt halmaz kinyerése...
Tesztelésre kiválasztva: 500 dal.

4. Kiértékelés futtatása...


Dalok tesztelése: 100%|██████████| 500/500 [02:30<00:00,  3.31it/s]



🏆 KIÉRTÉKELÉS EREDMÉNYE 🏆

--- TOP 10 AJÁNLÁS ---
BASELINE (Tiszta Word2Vec - Felső korlát):
  Hit Rate@10:  98.60%
  Precision@10: 85.00%
  Recall@10:    0.13%

CNN MODELL (Audio alapján generált vektor):
  Hit Rate@10:  56.20%
  Precision@10: 19.34%
  Recall@10:    0.01%

--- TOP 20 AJÁNLÁS ---
BASELINE (Tiszta Word2Vec - Felső korlát):
  Hit Rate@20:  99.40%
  Precision@20: 81.43%
  Recall@20:    0.23%

CNN MODELL (Audio alapján generált vektor):
  Hit Rate@20:  65.60%
  Precision@20: 19.06%
  Recall@20:    0.03%


In [1]:
import h5py
import numpy as np
import tensorflow as tf
from gensim.models import Word2Vec
from collections import defaultdict
from tqdm import tqdm # Folyamatjelzőhöz: pip install tqdm

# --- ÚTVONALAK ÉS BEÁLLÍTÁSOK ---
H5_PATH = "../Dataset/spotify_dataset.h5"
W2V_PATH = "../Models/song2vec.model"
CNN_PATH = "../Models/spotify_cnn_model.keras" # Vagy a pontos elérési út
K_VALUES = [10, 20] # Top 10 és Top 20 találatot is vizsgálunk
NUM_TEST_SAMPLES = 500 # Kezdésnek csak 500 dalon teszteljünk, hogy lássuk, lefut-e! (Állítsd None-ra a teljes teszthez)

# Custom rétegek és loss a CNN betöltéséhez
class L2NormLayer(tf.keras.layers.Layer):
    def call(self, x):
        return tf.math.l2_normalize(x, axis=1)

def cosine_loss(y_true, y_pred):
    return 1.0 - tf.reduce_mean(tf.reduce_sum(tf.math.l2_normalize(y_true, axis=1) * y_pred, axis=1))

def calculate_metrics(recommended_uris, relevant_uris, K):
    """Kiszámolja a metrikákat egy adott K értékre."""
    top_k = recommended_uris[:K]
    hits = len(set(top_k).intersection(relevant_uris))
    
    hit_rate = 1 if hits > 0 else 0
    precision = hits / K
    recall = hits / len(relevant_uris) if len(relevant_uris) > 0 else 0
    
    return hit_rate, precision, recall

def main():
    print("1. Modellek betöltése...")
    w2v_model = Word2Vec.load(W2V_PATH)
    cnn_model = tf.keras.models.load_model(
        CNN_PATH, custom_objects={'cosine_loss': cosine_loss, 'L2NormLayer': L2NormLayer}
    )

    print("\n2. Ground Truth szótárak építése a RAM-ban (ez eltarthat 1-2 percig)...")
    track_to_playlists = defaultdict(set)
    playlist_to_tracks = defaultdict(set)
    
    with h5py.File(H5_PATH, "r") as hf:
        pids = hf["playlist_tracks/pid"][:]
        uris = hf["playlist_tracks/track_uri"][:]
        
        for i in tqdm(range(len(pids)), desc="Kapcsolatok feldolgozása"):
            pid = pids[i]
            uri = uris[i].decode('utf-8') if isinstance(uris[i], bytes) else uris[i]
            
            track_to_playlists[uri].add(pid)
            playlist_to_tracks[pid].add(uri)

    print("\n3. Teszt halmaz kinyerése...")
    with h5py.File(H5_PATH, "r") as hf:
        splits = hf["ml/split"][:]
        test_indices = [i for i, s in enumerate(splits) if (s.decode('utf-8') if isinstance(s, bytes) else s) == "test"]
        
        if NUM_TEST_SAMPLES and len(test_indices) > NUM_TEST_SAMPLES:
            np.random.seed(42)
            test_indices = np.random.choice(test_indices, NUM_TEST_SAMPLES, replace=False)
            
        print(f"Tesztelésre kiválasztva: {len(test_indices)} dal.")

        # Metrika gyűjtők
        results = {"baseline": {k: {"hr": [], "prec": [], "rec": []} for k in K_VALUES},
                   "cnn":      {k: {"hr": [], "prec": [], "rec": []} for k in K_VALUES}}

        print("\n4. Kiértékelés futtatása...")
        for idx in tqdm(test_indices, desc="Dalok tesztelése"):
            uri = hf["ml/track_uri"][idx]
            uri = uri.decode('utf-8') if isinstance(uri, bytes) else uri
            
            # Csak azokat tudjuk tesztelni, amik benne vannak a W2V szótárban
            if uri not in w2v_model.wv:
                continue
                
            # --- GROUND TRUTH KINYERÉSE ---
            pids = track_to_playlists.get(uri, set())
            if not pids: 
                continue # Ha nincs egy listán sem, nem tudjuk tesztelni
                
            relevant_uris = set()
            for pid in pids:
                relevant_uris.update(playlist_to_tracks[pid])
            relevant_uris.discard(uri) # Magát a query dalt kivesszük a relevánsak közül
            
            if not relevant_uris:
                continue

            # --- A) BASELINE: Sima Word2Vec predikció ---
            # Bekérjük a legközelebbi szomszédokat a w2v térből
            w2v_similars = w2v_model.wv.most_similar(uri, topn=max(K_VALUES))
            w2v_recs = [u for u, score in w2v_similars]

            # --- B) CNN MODELL PREDIKCIÓ ---
            # Kinyerjük az audió feature-öket (hozzáadjuk a batch és channel dimenziókat: 1, X, Y, 1)
            mel = np.expand_dims(np.expand_dims(hf["spectrograms/mel"][idx], axis=0), axis=-1)
            chroma = np.expand_dims(np.expand_dims(hf["spectrograms/chroma"][idx], axis=0), axis=-1)
            tempo = np.expand_dims(np.expand_dims(hf["spectrograms/tempogram"][idx], axis=0), axis=-1)
            
            pred_vector = cnn_model.predict([mel, chroma, tempo], verbose=0)[0]
            
            # Megkeressük a prediktált vektorhoz legközelebbi dalokat a W2V szótárban
            cnn_similars = w2v_model.wv.similar_by_vector(pred_vector, topn=max(K_VALUES))
            # Kivesszük magát a dalt az ajánlásokból, ha véletlenül azt találná meg
            cnn_recs = [u for u, score in cnn_similars if u != uri][:max(K_VALUES)]
            
            # Előfordulhat, hogy a szűrés miatt 1-el rövidebb a lista, pótoljuk ha kell
            if len(cnn_recs) < max(K_VALUES):
                extra = w2v_model.wv.similar_by_vector(pred_vector, topn=max(K_VALUES)+1)
                cnn_recs = [u for u, score in extra if u != uri][:max(K_VALUES)]

            # --- METRIKÁK KISZÁMÍTÁSA ---
            for k in K_VALUES:
                # Baseline (W2V)
                hr_b, prec_b, rec_b = calculate_metrics(w2v_recs, relevant_uris, k)
                results["baseline"][k]["hr"].append(hr_b)
                results["baseline"][k]["prec"].append(prec_b)
                results["baseline"][k]["rec"].append(rec_b)
                
                # CNN
                hr_c, prec_c, rec_c = calculate_metrics(cnn_recs, relevant_uris, k)
                results["cnn"][k]["hr"].append(hr_c)
                results["cnn"][k]["prec"].append(prec_c)
                results["cnn"][k]["rec"].append(rec_c)

    # --- 5. EREDMÉNYEK KIÍRÁSA ---
    print("\n" + "="*50)
    print("🏆 KIÉRTÉKELÉS EREDMÉNYE 🏆")
    print("="*50)
    for k in K_VALUES:
        print(f"\n--- TOP {k} AJÁNLÁS ---")
        print("BASELINE (Tiszta Word2Vec - Felső korlát):")
        print(f"  Hit Rate@{k}:  {np.mean(results['baseline'][k]['hr'])*100:.2f}%")
        print(f"  Precision@{k}: {np.mean(results['baseline'][k]['prec'])*100:.2f}%")
        print(f"  Recall@{k}:    {np.mean(results['baseline'][k]['rec'])*100:.2f}%")
        
        print("\nCNN MODELL (Audio alapján generált vektor):")
        print(f"  Hit Rate@{k}:  {np.mean(results['cnn'][k]['hr'])*100:.2f}%")
        print(f"  Precision@{k}: {np.mean(results['cnn'][k]['prec'])*100:.2f}%")
        print(f"  Recall@{k}:    {np.mean(results['cnn'][k]['rec'])*100:.2f}%")

if __name__ == "__main__":
    main()

1. Modellek betöltése...


2. Ground Truth szótárak építése a RAM-ban (ez eltarthat 1-2 percig)...


Kapcsolatok feldolgozása: 100%|██████████| 66346428/66346428 [03:13<00:00, 343680.18it/s]



3. Teszt halmaz kinyerése...
Tesztelésre kiválasztva: 500 dal.

4. Kiértékelés futtatása...


Dalok tesztelése: 100%|██████████| 500/500 [02:06<00:00,  3.95it/s]



🏆 KIÉRTÉKELÉS EREDMÉNYE 🏆

--- TOP 10 AJÁNLÁS ---
BASELINE (Tiszta Word2Vec - Felső korlát):
  Hit Rate@10:  98.60%
  Precision@10: 85.00%
  Recall@10:    0.10%

CNN MODELL (Audio alapján generált vektor):
  Hit Rate@10:  56.20%
  Precision@10: 19.34%
  Recall@10:    0.01%

--- TOP 20 AJÁNLÁS ---
BASELINE (Tiszta Word2Vec - Felső korlát):
  Hit Rate@20:  99.40%
  Precision@20: 81.43%
  Recall@20:    0.19%

CNN MODELL (Audio alapján generált vektor):
  Hit Rate@20:  65.60%
  Precision@20: 19.06%
  Recall@20:    0.02%


In [ ]:
import h5py
import numpy as np
import tensorflow as tf
from gensim.models import Word2Vec
from collections import defaultdict
from tqdm import tqdm

# --- ÚTVONALAK ---
H5_PATH = "../Dataset/spotify_dataset.h5"
W2V_PATH = "../Models/song2vec.model"
CNN_PATH = "../Models/spotify_cnn_model.keras" # A te triple-input modelled

# --- BEÁLLÍTÁSOK ---
K_VALUE = 20 
NUM_TEST_PLAYLISTS = 500 # Kezdésnek 500 lista

# ==========================================
# 1. CUSTOM RÉTEGEK ÉS RANGSOROLÁSI METRIKÁK
# ==========================================
class L2NormLayer(tf.keras.layers.Layer):
    def call(self, x): return tf.math.l2_normalize(x, axis=1)

def cosine_loss(y_true, y_pred):
    return 1.0 - tf.reduce_mean(tf.reduce_sum(tf.math.l2_normalize(y_true, axis=1) * y_pred, axis=1))

def calculate_ranking_metrics(recommended_uris, relevant_uris, K):
    """HR, NDCG és AP (MAP-hez) számítása."""
    top_k = recommended_uris[:K]
    # Megnézzük melyik pozíción van találat (1 ha igen, 0 ha nem)
    hits = [1 if uri in relevant_uris else 0 for uri in top_k]
    
    # Hit Rate: volt-e legalább egy találat
    hr = 1 if sum(hits) > 0 else 0
    
    # Average Precision (AP) a MAP-hez
    precs = []
    num_hits = 0
    for i, hit in enumerate(hits):
        if hit:
            num_hits += 1
            precs.append(num_hits / (i + 1))
    ap = np.mean(precs) if precs else 0
    
    # NDCG számítása
    dcg = 0
    for i, hit in enumerate(hits):
        if hit:
            dcg += 1 / np.log2(i + 2)
    
    # Ideal DCG: ha az összes releváns dal a lista elején lenne
    idcg = 0
    for i in range(min(len(relevant_uris), K)):
        idcg += 1 / np.log2(i + 2)
    
    ndcg = dcg / idcg if idcg > 0 else 0
    
    return hr, ndcg, ap

# ==========================================
# 2. FŐ PROGRAM
# ==========================================
def main():
    print("1. Modellek betöltése...")
    w2v_model = Word2Vec.load(W2V_PATH)
    cnn_model = tf.keras.models.load_model(
        CNN_PATH, custom_objects={'cosine_loss': cosine_loss, 'L2NormLayer': L2NormLayer}
    )

    print("2. Adatbázis indexelése...")
    playlist_map = defaultdict(list)
    uri_to_h5_idx = {}
    
    with h5py.File(H5_PATH, "r") as hf:
        # URI -> H5 index leképzés a gyors kereséshez
        all_uris = hf["ml/track_uri"][:]
        for i, u in enumerate(all_uris):
            uri_str = u.decode('utf-8') if isinstance(u, bytes) else u
            uri_to_h5_idx[uri_str] = i
            
        # Playlistek felépítése
        pids = hf["playlist_tracks/pid"][:]
        track_uris = hf["playlist_tracks/track_uri"][:]
        for i in range(len(pids)):
            uri = track_uris[i].decode('utf-8') if isinstance(track_uris[i], bytes) else track_uris[i]
            playlist_map[pids[i]].append(uri)

    # 27.000 spektrogramos dal maszkolása
    # (Itt azt feltételezzük, hogy a spektrogram tábla nem üres soraiból tudjuk, mi van meg)
    print("3. Spektrogram-elérhetőség ellenőrzése...")
    with h5py.File(H5_PATH, "r") as hf:
        # Egy gyors mintavételezés a Mel táblából, hogy tudjuk melyik indexek érvényesek
        # Ha van has_spectrogram oszlopod, használd azt! 
        # Ha nincs, akkor pl. megnézzük a Mel spektrogramok szórását (ahol > 0, ott van adat)
        has_spec = np.any(hf["spectrograms/mel"][:], axis=(1, 2))

    # Tesztelésre alkalmas listák (Shuffle)
    all_pids = list(playlist_map.keys())
    np.random.shuffle(all_pids)

    results = {"hr": [], "ndcg": [], "map": []}
    tested_count = 0

    print(f"\n4. Szekvenciális kiértékelés indítása ({NUM_TEST_PLAYLISTS} lista)...")
    pbar = tqdm(total=NUM_TEST_PLAYLISTS)

    with h5py.File(H5_PATH, "r") as hf:
        for pid in all_pids:
            if tested_count >= NUM_TEST_PLAYLISTS: break
            
            songs = playlist_map[pid]
            if len(songs) < 3: continue # Túl rövid listákat kihagyjuk
            
            # --- VISSZAFELÉ KERESÉS (HORGONY) ---
            anchor_idx = -1
            # Utolsó előtti daltól (len-2) vissza az elejéig
            for i in range(len(songs) - 2, -1, -1):
                h5_idx = uri_to_h5_idx.get(songs[i])
                if h5_idx is not None and has_spec[h5_idx]:
                    anchor_idx = i
                    break
            
            if anchor_idx == -1: continue # Nincs spektrogramos dal a környezetben
            
            # --- KONTEXTUS ÉS CÉLPONT ---
            # Kontextus: a horgonyig bezárólag
            # Célpont: minden dal a horgony után
            context_uris = songs[:anchor_idx + 1]
            target_uris = set(songs[anchor_idx + 1:])
            
            vecs_to_average = []
            
            for i, uri in enumerate(context_uris):
                if i == anchor_idx:
                    # HORGONY: CNN predikció (Triple input: Mel, Chroma, Tempo)
                    idx = uri_to_h5_idx[uri]
                    m = np.expand_dims(np.expand_dims(hf["spectrograms/mel"][idx], axis=0), axis=-1).astype(np.float32)
                    c = np.expand_dims(np.expand_dims(hf["spectrograms/chroma"][idx], axis=0), axis=-1).astype(np.float32)
                    t = np.expand_dims(np.expand_dims(hf["spectrograms/tempogram"][idx], axis=0), axis=-1).astype(np.float32)
                    
                    cnn_vec = cnn_model.predict([m, c, t], verbose=0)[0]
                    vecs_to_average.append(cnn_vec)
                else:
                    # TÖBBI: Word2Vec vektor
                    if uri in w2v_model.wv:
                        vecs_to_average.append(w2v_model.wv[uri])
            
            if not vecs_to_average: continue
            
            # --- ÁTLAGOLT PROFIL VEKTOR ---
            profile_vec = np.mean(vecs_to_average, axis=0)
            
            # --- AJÁNLÁS ---
            # Kiszűrjük a már elhangzott dalokat
            sims = w2v_model.wv.similar_by_vector(profile_vec, topn=K_VALUE + len(context_uris))
            recs = [u for u, s in sims if u not in context_uris][:K_VALUE]
            
            # --- KIÉRTÉKELÉS ---
            hr, ndcg, ap = calculate_ranking_metrics(recs, target_uris, K_VALUE)
            results["hr"].append(hr)
            results["ndcg"].append(ndcg)
            results["map"].append(ap)
            
            tested_count += 1
            pbar.update(1)

    pbar.close()

    # --- EREDMÉNYEK KIÍRÁSA ---
    print("\n" + "="*50)
    print(f"🏆 SZEKVENCIÁLIS KIÉRTÉKELÉS EREDMÉNYE (K={K_VALUE})")
    print("="*50)
    print(f"Tesztelt listák száma: {tested_count}")
    print(f"Hit Rate@{K_VALUE}:  {np.mean(results['hr'])*100:.2f}%")
    print(f"NDCG@{K_VALUE}:      {np.mean(results['ndcg']):.4f}")
    print(f"MAP@{K_VALUE}:       {np.mean(results['map']):.4f}")
    print("="*50)

if __name__ == "__main__":
    main()

1. Modellek betöltése...

2. Adatbázis indexelése...
3. Spektrogram-elérhetőség ellenőrzése...

4. Szekvenciális kiértékelés indítása (500 lista)...


100%|██████████| 500/500 [01:17<00:00,  6.49it/s]



🏆 SZEKVENCIÁLIS KIÉRTÉKELÉS EREDMÉNYE (K=20)
Tesztelt listák száma: 500
Hit Rate@20:  2.80%
NDCG@20:      0.0063
MAP@20:       0.0078


## Más súlyozás

In [2]:
import h5py
import numpy as np
import tensorflow as tf
from gensim.models import Word2Vec
from collections import defaultdict
from tqdm import tqdm

# --- ÚTVONALAK ---
H5_PATH = "../Dataset/spotify_dataset.h5"
W2V_PATH = "../Models/song2vec.model"
CNN_PATH = "../Models/spotify_cnn_model.keras" # A te triple-input modelled

# --- BEÁLLÍTÁSOK ---
K_VALUE = 20 
NUM_TEST_PLAYLISTS = 500 # Kezdésnek 500 lista
CNN_BOOST = 2.0

# ==========================================
# 1. CUSTOM RÉTEGEK ÉS RANGSOROLÁSI METRIKÁK
# ==========================================
class L2NormLayer(tf.keras.layers.Layer):
    def call(self, x): return tf.math.l2_normalize(x, axis=1)

def cosine_loss(y_true, y_pred):
    return 1.0 - tf.reduce_mean(tf.reduce_sum(tf.math.l2_normalize(y_true, axis=1) * y_pred, axis=1))

def calculate_ranking_metrics(recommended_uris, relevant_uris, K):
    """HR, NDCG és AP (MAP-hez) számítása."""
    top_k = recommended_uris[:K]
    # Megnézzük melyik pozíción van találat (1 ha igen, 0 ha nem)
    hits = [1 if uri in relevant_uris else 0 for uri in top_k]
    
    # Hit Rate: volt-e legalább egy találat
    hr = 1 if sum(hits) > 0 else 0
    
    # Average Precision (AP) a MAP-hez
    precs = []
    num_hits = 0
    for i, hit in enumerate(hits):
        if hit:
            num_hits += 1
            precs.append(num_hits / (i + 1))
    ap = np.mean(precs) if precs else 0
    
    # NDCG számítása
    dcg = 0
    for i, hit in enumerate(hits):
        if hit:
            dcg += 1 / np.log2(i + 2)
    
    # Ideal DCG: ha az összes releváns dal a lista elején lenne
    idcg = 0
    for i in range(min(len(relevant_uris), K)):
        idcg += 1 / np.log2(i + 2)
    
    ndcg = dcg / idcg if idcg > 0 else 0
    
    return hr, ndcg, ap

# ==========================================
# 2. FŐ PROGRAM
# ==========================================
def main():
    print("1. Modellek betöltése...")
    w2v_model = Word2Vec.load(W2V_PATH)
    cnn_model = tf.keras.models.load_model(
        CNN_PATH, custom_objects={'cosine_loss': cosine_loss, 'L2NormLayer': L2NormLayer}
    )

    print("2. Adatbázis indexelése...")
    playlist_map = defaultdict(list)
    uri_to_h5_idx = {}
    
    with h5py.File(H5_PATH, "r") as hf:
        # URI -> H5 index leképzés a gyors kereséshez
        all_uris = hf["ml/track_uri"][:]
        for i, u in enumerate(all_uris):
            uri_str = u.decode('utf-8') if isinstance(u, bytes) else u
            uri_to_h5_idx[uri_str] = i
            
        # Playlistek felépítése
        pids = hf["playlist_tracks/pid"][:]
        track_uris = hf["playlist_tracks/track_uri"][:]
        for i in range(len(pids)):
            uri = track_uris[i].decode('utf-8') if isinstance(track_uris[i], bytes) else track_uris[i]
            playlist_map[pids[i]].append(uri)

    # 27.000 spektrogramos dal maszkolása
    # (Itt azt feltételezzük, hogy a spektrogram tábla nem üres soraiból tudjuk, mi van meg)
    print("3. Spektrogram-elérhetőség ellenőrzése...")
    with h5py.File(H5_PATH, "r") as hf:
        # Egy gyors mintavételezés a Mel táblából, hogy tudjuk melyik indexek érvényesek
        # Ha van has_spectrogram oszlopod, használd azt! 
        # Ha nincs, akkor pl. megnézzük a Mel spektrogramok szórását (ahol > 0, ott van adat)
        has_spec = np.any(hf["spectrograms/mel"][:], axis=(1, 2))

    # Tesztelésre alkalmas listák (Shuffle)
    all_pids = list(playlist_map.keys())
    np.random.shuffle(all_pids)

    results = {"hr": [], "ndcg": [], "map": []}
    tested_count = 0

    print(f"\n4. Szekvenciális kiértékelés indítása ({NUM_TEST_PLAYLISTS} lista)...")
    pbar = tqdm(total=NUM_TEST_PLAYLISTS)

    with h5py.File(H5_PATH, "r") as hf:
        for pid in all_pids:
            if tested_count >= NUM_TEST_PLAYLISTS: break
            
            songs = playlist_map[pid]
            if len(songs) < 3: continue 
            
            # Visszafelé keresés (Horgony)
            anchor_idx = -1
            for i in range(len(songs) - 2, -1, -1):
                h5_idx = uri_to_h5_idx.get(songs[i])
                if h5_idx is not None and has_spec[h5_idx]:
                    anchor_idx = i
                    break
            
            if anchor_idx == -1: continue 
            
            context_uris = songs[:anchor_idx + 1]
            target_uris = set(songs[anchor_idx + 1:])
            
            context_vectors = []
            weights = [] # [ÚJ] Súlyok gyűjtése
            
            for i, uri in enumerate(context_uris):
                # Alapsúly: minél nagyobb az i, annál közelebb van a horgonyhoz
                current_weight = i + 1 
                
                if i == anchor_idx:
                    # CNN Horgony predikció
                    idx = uri_to_h5_idx[uri]
                    m = np.expand_dims(np.expand_dims(hf["spectrograms/mel"][idx], axis=0), axis=-1).astype(np.float32)
                    c = np.expand_dims(np.expand_dims(hf["spectrograms/chroma"][idx], axis=0), axis=-1).astype(np.float32)
                    t = np.expand_dims(np.expand_dims(hf["spectrograms/tempogram"][idx], axis=0), axis=-1).astype(np.float32)
                    
                    cnn_vec = cnn_model.predict([m, c, t], verbose=0)[0]
                    context_vectors.append(cnn_vec)
                    # [ÚJ] A CNN horgony extra súlyt kap
                    weights.append(current_weight * CNN_BOOST)
                else:
                    if uri in w2v_model.wv:
                        context_vectors.append(w2v_model.wv[uri])
                        weights.append(current_weight)
            
            if not context_vectors: continue
            
            # --- [ÚJ] SÚLYOZOTT ÁTLAGOLÁS ---
            # Kiszámoljuk a súlyozott profil vektort
            profile_vec = np.average(context_vectors, axis=0, weights=weights)
            
            # --- AJÁNLÁS ÉS MÉRÉS ---
            sims = w2v_model.wv.similar_by_vector(profile_vec, topn=K_VALUE + len(context_uris))
            recs = [u for u, s in sims if u not in context_uris][:K_VALUE]
            
            hr, ndcg, ap = calculate_ranking_metrics(recs, target_uris, K_VALUE)
            results["hr"].append(hr)
            results["ndcg"].append(ndcg)
            results["map"].append(ap)
            
            tested_count += 1
            pbar.update(1)

    pbar.close()

    # --- EREDMÉNYEK KIÍRÁSA ---
    print("\n" + "="*50)
    print(f"🏆 SZEKVENCIÁLIS KIÉRTÉKELÉS EREDMÉNYE (K={K_VALUE})")
    print("="*50)
    print(f"Tesztelt listák száma: {tested_count}")
    print(f"Hit Rate@{K_VALUE}:  {np.mean(results['hr'])*100:.2f}%")
    print(f"NDCG@{K_VALUE}:      {np.mean(results['ndcg']):.4f}")
    print(f"MAP@{K_VALUE}:       {np.mean(results['map']):.4f}")
    print("="*50)

if __name__ == "__main__":
    main()

1. Modellek betöltése...
2. Adatbázis indexelése...
3. Spektrogram-elérhetőség ellenőrzése...

4. Szekvenciális kiértékelés indítása (500 lista)...


100%|██████████| 500/500 [01:26<00:00,  5.79it/s]



🏆 SZEKVENCIÁLIS KIÉRTÉKELÉS EREDMÉNYE (K=20)
Tesztelt listák száma: 500
Hit Rate@20:  2.60%
NDCG@20:      0.0077
MAP@20:       0.0090


In [1]:
pip install faiss-cpu

   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
    --------------------------------------- 0.3/18.9 MB ? eta -:--:--
   - -------------------------------------- 0.5/18.9 MB 8.5 MB/s eta 0:00:03
   -- ------------------------------------- 1.3/18.9 MB 4.2 MB/s eta 0:00:05
   ---- ----------------------------------- 2.1/18.9 MB 3.7 MB/s eta 0:00:05
   ---- ----------------------------------- 2.1/18.9 MB 3.7 MB/s eta 0:00:05
   ------ --------------------------------- 3.1/18.9 MB 3.2 MB/s eta 0:00:05
   ------------- -------------------------- 6.6/18.9 MB 5.0 MB/s eta 0:00:03
   ----------------------- ---------------- 11.0/18.9 MB 7.2 MB/s eta 0:00:02
   ----------------------------------- ---- 16.8/18.9 MB 9.7 MB/s eta 0:00:01
   ---------------------------------------- 18.9/18.9 MB 10.3 MB/s  0:00:02
Note: you may need to rest

In [5]:
import numpy as np
import faiss
from tqdm import tqdm
import math

# --- KONFIGURÁCIÓ ---
HYBRID_MATRIX_PATH = "../Models/hybrid_embedding_matrix.npy"
TEST_PIDS_PATH = "../Models/test_pids.npy"
K = 50  # Top-K ajánlás

def calculate_ndcg(rank, k):
    """NDCG kiszámítása (mivel 1 releváns elemünk van, az IDCG = 1)"""
    if rank < k:
        return 1.0 / math.log2(rank + 2) # rank+2 mert 0-tól indexelünk
    return 0

def calculate_map(rank, k):
    """MAP kiszámítása (1 releváns elem esetén = 1/(rank+1))"""
    if rank < k:
        return 1.0 / (rank + 1)
    return 0

# 1. Adatok betöltése
print("📂 Adatok betöltése...")
hybrid_matrix = np.load(HYBRID_MATRIX_PATH).astype('float32')
test_playlists = np.load(TEST_PIDS_PATH, allow_pickle=True)

# Normalizálás a koszinusz-hasonlósághoz
print("⚡ Vektorok normalizálása...")
faiss.normalize_L2(hybrid_matrix)

# 2. FAISS Index felépítése
print("🏗️ FAISS index építése...")
d = hybrid_matrix.shape[1]
index = faiss.IndexFlatIP(d) 
index.add(hybrid_matrix)

# 3. Kiértékelés
hr_count = 0
map_sum = 0
ndcg_sum = 0
total_playlists = 0

print(f"🧪 Kiértékelés futtatása {len(test_playlists):,} listán...")

for pl in tqdm(test_playlists[::10]):
    if len(pl) < 2:
        continue
    
    # Utolsó elem a cél, előtte lévők a kontextus
    context_indices = pl[:-1]
    target_idx = pl[-1]
    
    # --- SÚLYOZOTT CENTROID ---
    vectors = hybrid_matrix[context_indices]
    weights = np.arange(1, len(context_indices) + 1).astype('float32').reshape(-1, 1)
    
    weighted_centroid = np.sum(vectors * weights, axis=0) / np.sum(weights)
    
    # Keresés előtti normalizálás
    centroid_norm = weighted_centroid / (np.linalg.norm(weighted_centroid) + 1e-8)
    centroid_norm = centroid_norm.reshape(1, -1).astype('float32')

    # --- KERESÉS (FAISS) ---
    # Többet kérünk le, hogy a már hallgatottakat ki tudjuk szűrni
    D, I = index.search(centroid_norm, K + len(context_indices))
    
    # Ajánlások szűrése (ne ajánljuk azt, ami már a listában van)
    recommendations = []
    for idx in I[0]:
        if idx not in context_indices:
            recommendations.append(idx)
        if len(recommendations) == K:
            break

    # --- METRIKÁK SZÁMÍTÁSA ---
    total_playlists += 1
    if target_idx in recommendations:
        rank = recommendations.index(target_idx)
        
        hr_count += 1
        map_sum += calculate_map(rank, K)
        ndcg_sum += calculate_ndcg(rank, K)

# 4. Eredmények összesítése
final_hr = hr_count / total_playlists
final_map = map_sum / total_playlists
final_ndcg = ndcg_sum / total_playlists

print("\n" + "="*40)
print(f"📊 BASELINE EREDMÉNYEK (Top-{K})")
print(f"Hit Rate (HR@{K}):  {final_hr:.4f}")
print(f"MAP@{K}:           {final_map:.4f}")
print(f"NDCG@{K}:          {final_ndcg:.4f}")
print("="*40)

📂 Adatok betöltése...
⚡ Vektorok normalizálása...
🏗️ FAISS index építése...
🧪 Kiértékelés futtatása 99,180 listán...


100%|██████████| 9918/9918 [05:38<00:00, 29.33it/s]


📊 BASELINE EREDMÉNYEK (Top-50)
Hit Rate (HR@50):  0.0322
MAP@50:           0.0053
NDCG@50:          0.0104
